# Phase 1b: score Shrutilipi clips with IndicWhisper (free Colab T4)

1. **Runtime → Change runtime type → T4 GPU.**
2. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) with a Hugging Face **write** token, and enable notebook access for it.
3. Accept the [Shrutilipi terms](https://huggingface.co/datasets/ai4bharat/Shrutilipi) with the same HF account.
4. Set `LANG`, then **Runtime → Run all**.

Scores upload to the Hub every few shards. If the session disconnects, reconnect and **Run all** again: finished shards are skipped.
Every cell stops with an error if its command fails, so the first red cell is the real problem.

In [ ]:
LANG = "gujarati"  # or "hindi"
SCORES_REPO = "akashkuushwahaa/shrutilipi-gu-hi-scores"  # private dataset, created if missing

import os, subprocess, sys

def run(cmd):
    """Like `!cmd`, but raises when the command fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # raises if the secret is missing or not shared
import shutil
if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU in this session. Runtime -> Change runtime type -> T4 GPU -> Save, then Run all. "
                     "If T4 is greyed out or Colab says it cannot connect to a GPU backend, the free GPU quota "
                     "is used up: try again later (results so far are safe on the Hub).")
run("nvidia-smi --query-gpu=name,memory.total --format=csv")  # fails if no GPU runtime
run("df -h /content | tail -1")
run("pip install -q --upgrade 'git+https://github.com/akashkuushwahaa/indic-whisper'")
run("hf auth whoami")

In [ ]:
run(f"iw-score-job --lang {LANG} --scores-repo {SCORES_REPO} --work /content/iw --gpus 1")

## Quick look at the scores

In [ ]:
import glob, pandas as pd
files = glob.glob(f"/content/iw/data/{LANG}/scores/*/*.parquet")
if not files:
    raise SystemExit("No score files yet: the scoring cell above didn't finish any shard. Check its output.")
s = pd.concat(pd.read_parquet(f) for f in files)
print(f"{len(s):,} clips scored from {len(files)} shards")
print(s.cer.describe(percentiles=[.5, .75, .9, .95, .99]))
for t in (0.2, 0.3, 0.4, 0.5):
    print(f"CER > {t}: {(s.cer > t).mean():.1%} of clips would be dropped")